# Transfer Learning com Deep Learning — Gatos vs Cachorros

**Desafio DIO — Trilha de Deep Learning / Transfer Learning**

Autor: José Wagner Blanco Júnior — Consultoria & Mentoria Blanco

Este notebook aplica a técnica de **Transfer Learning** utilizando a arquitetura **MobileNetV2**
(pré-treinada no ImageNet) para resolver um problema de classificação binária de imagens:
**gatos vs cachorros**, a partir do dataset `cats_vs_dogs` (TensorFlow Datasets).

Fluxo do notebook:
1. Carregamento do dataset via `tensorflow_datasets`
2. Pré-processamento e data augmentation
3. Construção do modelo base congelado (feature extraction)
4. Treinamento da "cabeça" de classificação
5. Fine-tuning das últimas camadas do modelo base
6. Avaliação, métricas e gráficos de acurácia/loss
7. Inferência em imagens novas
8. Exportação do modelo treinado (`.keras`)


In [ ]:
# 1. Instalação e imports
!pip -q install tensorflow tensorflow-datasets matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

print("TensorFlow:", tf.__version__)
print("GPUs disponíveis:", tf.config.list_physical_devices('GPU'))


## 2. Carregamento do dataset

Utilizamos o dataset `cats_vs_dogs` do TensorFlow Datasets, que contém ~23.000 imagens
de gatos e cachorros (duas classes). Como o dataset não vem com split de validação/teste
oficial, criamos os splits manualmente (80% treino / 10% validação / 10% teste).


In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32

(raw_train, raw_val, raw_test), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

num_classes = metadata.features['label'].num_classes
class_names = metadata.features['label'].names
print("Classes:", class_names)

# Visualizando algumas amostras
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(raw_train.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[label.numpy()])
    plt.axis('off')
plt.suptitle('Amostras do dataset cats_vs_dogs')
plt.show()


## 3. Pré-processamento e data augmentation

Redimensionamos as imagens para 160x160 (tamanho de entrada esperado pelo MobileNetV2)
e aplicamos normalização + augmentation (flip horizontal e rotação leve) apenas no treino.


In [ ]:
def format_image(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
])

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

train_ds = (raw_train
            .map(format_image)
            .shuffle(1000)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (raw_val
          .map(format_image)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

test_ds = (raw_test
           .map(format_image)
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE))


## 4. Construção do modelo base (Transfer Learning — Feature Extraction)

Carregamos o **MobileNetV2** pré-treinado no ImageNet, sem a camada de topo (`include_top=False`),
e congelamos seus pesos. Sobre ele, adicionamos uma cabeça de classificação própria para o
nosso problema binário (gato vs cachorro).


In [ ]:
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Congela o modelo base

inputs = tf.keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1)(x)  # logit para classificação binária
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.summary()


## 5. Treinamento da cabeça de classificação

Treinamos apenas as camadas novas (o modelo base permanece congelado nesta etapa).


In [ ]:
EPOCHS_HEAD = 8

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD
)


## 6. Fine-tuning

Descongelamos as últimas camadas do MobileNetV2 e continuamos o treinamento com uma
taxa de aprendizado bem menor, para ajustar finamente os pesos ao nosso domínio
(gatos vs cachorros) sem destruir o que já foi aprendido no ImageNet.


In [ ]:
base_model.trainable = True

# Congela todas as camadas antes da camada 100 (mantém features genéricas)
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)

EPOCHS_FINE_TUNE = 6
total_epochs = EPOCHS_HEAD + EPOCHS_FINE_TUNE

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1
)


## 7. Avaliação e gráficos de acurácia/loss

In [ ]:
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Treino')
plt.plot(val_acc, label='Validação')
plt.axvline(EPOCHS_HEAD - 1, color='gray', linestyle='--', label='Início do fine-tuning')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Treino')
plt.plot(val_loss, label='Validação')
plt.axvline(EPOCHS_HEAD - 1, color='gray', linestyle='--', label='Início do fine-tuning')
plt.legend(loc='upper right')
plt.title('Loss de Treino e Validação')

plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")
print(f"Loss no conjunto de teste: {test_loss:.4f}")


## 8. Inferência em novas imagens

Pegamos um lote do conjunto de teste e comparamos a predição do modelo com o rótulo real.


In [ ]:
image_batch, label_batch = next(iter(test_ds))
predictions = model.predict(image_batch)
predictions = tf.nn.sigmoid(predictions).numpy().flatten()
predicted_labels = (predictions > 0.5).astype(int)

plt.figure(figsize=(12, 12))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image_batch[i].numpy().astype('uint8'))
    real = class_names[label_batch[i].numpy()]
    pred = class_names[predicted_labels[i]]
    color = 'green' if real == pred else 'red'
    plt.title(f"Real: {real} | Previsto: {pred}", color=color)
    plt.axis('off')
plt.suptitle('Predições do modelo no conjunto de teste')
plt.savefig('inference_examples.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Exportação do modelo

Salvamos o modelo treinado em formato `.keras`, pronto para reuso ou deploy.


In [ ]:
model.save('transfer_learning_cats_vs_dogs.keras')
print("Modelo salvo com sucesso!")


## 10. Conclusão

Com Transfer Learning a partir do MobileNetV2, foi possível atingir alta acurácia na
classificação de gatos vs cachorros com um volume de treinamento muito menor do que
seria necessário treinando uma rede do zero, aproveitando o conhecimento visual já
aprendido no ImageNet.

**Próximos passos possíveis:**
- Testar outras arquiteturas base (EfficientNet, ResNet50, VGG16)
- Aplicar o mesmo pipeline a um dataset próprio (fotos pessoais, animais de estimação, etc.)
- Servir o modelo via API (FastAPI/Flask) para inferência em produção
